IMPORTS

In [14]:
import pandas as pd 
import numpy as np 

LOAD RAW DATA

In [15]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Initial Shape:", df.shape)
df.head()

Initial Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


DATA CLEANING

In [16]:
# Fix TotalCharges
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Fill missing values
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID
df.drop(columns=['customerID'], inplace=True)

print("After Cleaning Shape:", df.shape)

After Cleaning Shape: (7043, 20)


C:\Users\mehta\AppData\Local\Temp\ipykernel_16420\4250680919.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


TENURE GROUP FEATURE

In [17]:
def tenure_group(x):
    if x <=12:
        return "New"
    elif x <= 36:
        return "Mid"
    else:
        return "Loyal"
    
df['tenure_group'] = df['tenure'].apply(tenure_group)

df['tenure_group'].value_counts()

tenure_group
Loyal    3001
New      2186
Mid      1856
Name: count, dtype: int64

SERVICE COUNT FEATURE

In [18]:
services = [
    'PhoneService','MultipleLines','InternetService',
    'OnlineSecurity','OnlineBackup','DeviceProtection',
    'TechSupport','StreamingTV','StreamingMovies'
]

df['num_services'] = df[services].apply(lambda x: sum(x == "Yes"),axis=1)

df[['num_services']].head()

,num_services
0,1
1,3
2,3
3,3
4,1


CONTRACT RISK FEATURE

In [19]:
contract_map = {
    'Month-to-month': 2,
    'One year': 1,
    'Two year': 0
}

df['contract_risk'] = df['Contract'].map(contract_map)
df[['Contract', 'contract_risk']].head()

,Contract,contract_risk
0,Month-to-month,2
1,One year,1
2,Month-to-month,2
3,One year,1
4,Month-to-month,2


ENGAGEMENT SCORE

In [20]:
df['engagement_score'] = (
    df['num_services'] +
    df['tenure'] / 12+
    (df['MonthlyCharges']/100)
)

df[['engagement_score']].head()

,engagement_score
0,1.381833
1,6.402833
2,3.705167
3,7.173000
4,1.873667


In [21]:
print("Final Shape:", df.shape)
df.head()

Final Shape: (7043, 24)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,num_services,contract_risk,engagement_score
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,New,1,2,1.381833
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,Mid,3,1,6.402833
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,New,3,2,3.705167
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,Loyal,3,1,7.173000
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,New,1,2,1.873667


In [22]:
df.to_csv("../data/processed/churn_processed.csv", index=False)

print(" Processed dataset saved successfully!")

 Processed dataset saved successfully!
